In [ ]:
# %%capture
%matplotlib inline

import os
import warnings
import logging
import time
from pathlib import Path
from typing import Optional, List, Dict

# --- Silence TensorFlow and add-ons warnings for a clean log ---
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
os.environ['ABSL_LOG_LEVEL'] = '3'
warnings.filterwarnings('ignore', category=UserWarning, module='tensorflow_addons')
import tensorflow as tf
tf.get_logger().setLevel('ERROR')

# --- Project Imports ---
from forecast_pipeline.config import LOG_LEVEL, DEFAULT_EXP_PARAMS, MAX_WORKERS, DEFAULT_DATASET
from forecast_pipeline.jobs import generate_jobs, select_data_sources, create_filter_configurations
from forecast_pipeline.metrics import clean_and_structure_results
from forecast_pipeline.io_utils import generate_experiment_name, save_experiment_to_excel, configure_logging
from forecast_pipeline.runner import run_experiments_for_config
from common.config_wells import DATA_SOURCES

In [ ]:
# --- Main Pipeline Function ---
def main_legacy(
    ensemble_models: int = 1,
    filter_methods: Optional[List] = None,
    selected_sources: Optional[List] = None,
) -> Dict:
    """
    End-to-end pipeline for the legacy run mode.
    Selects data sources, iterates configs, runs jobs, and collects results.
    """
    logging.info("Starting legacy pipeline...")
    sources = select_data_sources(DATA_SOURCES, selected_sources)
    if not sources:
        logging.warning("No data sources selected; exiting.")
        return {}

    results = {}
    for cfg in create_filter_configurations(filter_methods):
        # Legacy runner always uses in-memory results collation
        results.update(run_experiments_for_config(
            cfg, sources, ensemble_models, profile_path=None
        ))

    logging.info("Legacy pipeline finished.")
    return results

# --- Entry Point ---
def run_legacy_pipeline():

    configure_logging()
    ensemble_size = 1
    start_time = time.time()

    # Run pipeline
    results = main_legacy(
        ensemble_models=ensemble_size,
        filter_methods=None,
        selected_sources=DEFAULT_DATASET,
    )

    # --- Result Saving ---
    exp_name = generate_experiment_name(
        DEFAULT_DATASET,
        DEFAULT_EXP_PARAMS["architecture_name"],
        ensemble_size
    )
    output_path = save_experiment_to_excel(
        DEFAULT_EXP_PARAMS,
        results,
        exp_name,
        DEFAULT_DATASET,
        ensemble_size,
    )
    logging.info(f"Results saved to: {output_path}")

    elapsed_time = time.time() - start_time
    logging.info(f"Execution time: {elapsed_time:.2f} seconds")

# --- Run if script is called directly ---
if __name__ == "__main__":
    run_legacy_pipeline()
